# Notebook 02: Cruce de Datos y Creación del DataFrame Maestro

**Objetivos de este notebook:**
1. Cargar las tablas base (`personas_seleccionadas`, `d_capitulos`, `d2_capitulos`) que contienen la demografía.
2. Hacer un *Inner Join* entre las tablas base para tener nuestra "población objetivo".
3. Cargar las tablas de sustancias (Capítulos E hasta S) y hacer *Left Joins* contra la tabla base.
4. Tratar las columnas duplicadas (sufijos `_x` y `_y`).
5. Generar la "Tabla Maestra" (`df_master.parquet`) lista para el Análisis Exploratorio de Datos (EDA).

**Contexto de los Nulos (NaNs):** 
Al hacer *Left Joins*, las personas que NO consumen una sustancia (ej. Cocaína) tendrán valores nulos en todas las columnas de ese capítulo (ej. L_01, L_02). En la ENCSPA esto es correcto; un nulo aquí significa "Salto de pregunta por no consumo".

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


In [10]:
# Rutas relativas asumiendo que estamos en la carpeta /notebooks/
BASE_DIR = Path('..')
INTERIM_DIR = BASE_DIR / 'data' / 'interim'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

# Crear la carpeta processed si no existe
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Las llaves únicas que identifican a una persona específica en un hogar
LLAVES = ['directorio', 'secuencia_encuesta', 'secuencia_p', 'orden']

print(f"Buscando archivos en: {INTERIM_DIR}")

Buscando archivos en: ..\data\interim


In [11]:
# 1. Cargar la tabla principal (La persona seleccionada aleatoriamente en el hogar)
df_base = pd.read_parquet(INTERIM_DIR / 'personas_seleccionadas.parquet')
print(f"Filas en personas_seleccionadas: {df_base.shape[0]}")

# 2. Cargar Capítulos D y D2 (Características generales, educación, salud, identidad)
df_d = pd.read_parquet(INTERIM_DIR / 'd_capitulos.parquet')
df_d2 = pd.read_parquet(INTERIM_DIR / 'd2_capitulos.parquet')

# Función para evitar columnas duplicadas al unir (ej. que FEX_C no se vuelva FEX_C_x y FEX_C_y)
def drop_overlapping_cols(df_left, df_right, keys):
    overlap = [col for col in df_right.columns if col in df_left.columns and col not in keys]
    return df_right.drop(columns=overlap)

# 3. Unir las tablas base usando Inner Join
# Usamos Inner Join porque todo encuestado DEBE tener información demográfica
df_d_clean = drop_overlapping_cols(df_base, df_d, LLAVES)
df_master = pd.merge(df_base, df_d_clean, on=LLAVES, how='inner')

df_d2_clean = drop_overlapping_cols(df_master, df_d2, LLAVES)
df_master = pd.merge(df_master, df_d2_clean, on=LLAVES, how='inner')

print(f"Filas en la Base Demográfica consolidada: {df_master.shape[0]}")
print(f"Columnas hasta ahora: {df_master.shape[1]}")

Filas en personas_seleccionadas: 49756
Filas en la Base Demográfica consolidada: 49756
Columnas hasta ahora: 49


In [12]:
# Lista de los capítulos de consumo que queremos agregar
capitulos_sustancias = [
    'e_capitulos', # Tabaco
    'f_capitulos', # Alcohol
    'g_capitulos', # Filtro Drogas Ilegales
    'h_capitulos', # Tranquilizantes
    'i_capitulos', # Estimulantes
    'j_capitulos', # Inhalables
    'k_capitulos', # Marihuana
    'l_capitulos', # Cocaína
    'm_capitulos', # Basuco
    'n_capitulos', # Éxtasis
    'o_capitulos', # Heroína
    'p_capitulos', # Otras sustancias (LSD, Hongos, etc.)
    'q_capitulos', # Tratamiento
    'r_capitulos', # Trabajo
    's_capitulos'  # Embarazo
]

print("Iniciando cruce de módulos de sustancias (Left Joins)...")

for cap in capitulos_sustancias:
    file_path = INTERIM_DIR / f"{cap}.parquet"
    
    if file_path.exists():
        df_cap = pd.read_parquet(file_path)
        
        # Eliminar columnas repetidas para no ensuciar el DataFrame
        df_cap_clean = drop_overlapping_cols(df_master, df_cap, LLAVES)
        
        # LEFT JOIN: Conservamos todos los de df_master, si no consumen, quedará NaN
        df_master = pd.merge(df_master, df_cap_clean, on=LLAVES, how='left')
        print(f" ✅ Unido {cap}. Total columnas: {df_master.shape[1]}")
    else:
        print(f" ⚠️ Archivo no encontrado: {cap}.parquet")

print(f"\nCruce finalizado. Filas: {df_master.shape[0]}, Columnas: {df_master.shape[1]}")

Iniciando cruce de módulos de sustancias (Left Joins)...
 ✅ Unido e_capitulos. Total columnas: 61
 ✅ Unido f_capitulos. Total columnas: 94
 ✅ Unido g_capitulos. Total columnas: 188
 ✅ Unido h_capitulos. Total columnas: 218
 ✅ Unido i_capitulos. Total columnas: 240
 ✅ Unido j_capitulos. Total columnas: 265
 ✅ Unido k_capitulos. Total columnas: 309
 ✅ Unido l_capitulos. Total columnas: 346
 ✅ Unido m_capitulos. Total columnas: 380
 ✅ Unido n_capitulos. Total columnas: 401
 ✅ Unido o_capitulos. Total columnas: 425
 ✅ Unido p_capitulos. Total columnas: 431
 ✅ Unido q_capitulos. Total columnas: 449
 ✅ Unido r_capitulos. Total columnas: 461
 ✅ Unido s_capitulos. Total columnas: 474

Cruce finalizado. Filas: 49756, Columnas: 474


In [13]:
# Crear diccionarios básicos según el manual del DANE
dicc_sexo = {1: 'Hombre', 2: 'Mujer'}
dicc_si_no = {1: 'Sí', 2: 'No', 9: 'No sabe / No responde'}
dicc_estado_civil = {
    1: 'No está casado(a) y vive en pareja < 2 años',
    2: 'No está casado(a) y vive en pareja >= 2 años',
    3: 'Está casado(a)',
    4: 'Está viudo(a)',
    5: 'Está separado(a) o divorciado(a)',
    6: 'Está soltero(a)'
}
dicc_educacion = {
    1: 'Ninguno', 2: 'Preescolar', 3: 'Básica primaria',
    4: 'Básica secundaria', 5: 'Media', 6: 'Técnica/Tecnológica',
    7: 'Universitaria', 8: 'Postgrado', 9: 'No sabe / No informa'
}

# Aplicar los diccionarios a variables clave
# Mapeo de sexo (la columna original suele llamarse 'sexo' o 'p6020', revisa tu base, aquí asumo 'sexo' basado en el diccionario DANE)
if 'sexo' in df_master.columns:
    df_master['sexo_desc'] = df_master['sexo'].map(dicc_sexo)

# Estado civil (Variable D2_03)
if 'd2_03' in df_master.columns:
    df_master['estado_civil_desc'] = df_master['d2_03'].map(dicc_estado_civil)

# Nivel educativo (Variable D2_05)
if 'd2_05' in df_master.columns:
    df_master['nivel_educativo'] = df_master['d2_05'].map(dicc_educacion)

# Preguntas filtro de consumo alguna vez en la vida (Transformar a Sí/No/NaN)
# E_01: Tabaco, F_03: Alcohol, K_03: Marihuana (12 meses, asumo como proxy), L_02: Cocaína (12 meses)
filtros = {
    'e_01': 'consumo_tabaco_vida',
    'f_03': 'consumo_alcohol_vida',
    'k_03': 'consumo_marihuana_12m',
    'l_02': 'consumo_cocaina_12m'
}

for col_original, col_nueva in filtros.items():
    if col_original in df_master.columns:
        df_master[col_nueva] = df_master[col_original].map(dicc_si_no)
        # Llenar los NaNs generados por el Left Join indicando que no consumen o no reportaron
        df_master[col_nueva] = df_master[col_nueva].fillna('No consumió / No llegó al módulo')

print("Mapeo de variables clave completado.")

Mapeo de variables clave completado.


In [14]:
# Vamos a crear una columna valiosa: Rango de Edad
if 'edad' in df_master.columns:
    bins = [11, 17, 24, 34, 44, 54, 65]
    labels = ['12-17', '18-24', '25-34', '35-44', '45-54', '55-65']
    df_master['rango_edad'] = pd.cut(df_master['edad'], bins=bins, labels=labels)
    
print("Variables sintéticas creadas.")

Variables sintéticas creadas.


In [15]:
# Guardar el DataFrame final consolidado
path_export = PROCESSED_DIR / 'df_master.parquet'
df_master.to_parquet(path_export, index=False)

print(f" El DataFrame maestro ha sido guardado en: {path_export}")
print(f"Tamaño final en memoria: {df_master.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

 El DataFrame maestro ha sido guardado en: ..\data\processed\df_master.parquet
Tamaño final en memoria: 220.13 MB
